# Benchmark monthly cross-sectional scaling

This notebook benchmarks `scale_chars_cross_sectionally_by_month_bigdata_v2` against `scale_chars_cross_sectionally_by_month`.

The synthetic panel contains 60 years of monthly observations and `n_firms_per_month` cross-sectional observations per month. A single random characteristic is duplicated: each implementation transforms one copy.

`date` has dtype `period[M]`, so every unique date value is exactly one monthly cross-section.

In [5]:
import gc
import time
from statistics import median

import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer


In [6]:
def scale_chars_cross_sectionally_by_month(
    df: pd.DataFrame,
    characteristic_cols: list[str],
    n_quantiles: int = 1000,
    random_state: int = 42,
):
    """
    Scale characteristics cross-sectionally month by month usinf the QuantileTransformer into [-1,1].
    """

    
    out = df.copy()

    qt = QuantileTransformer(
        n_quantiles=n_quantiles,
        output_distribution="uniform",
        random_state=random_state,
        subsample=10000,
    )

    for col in characteristic_cols:
        lam = lambda x: qt.fit_transform(x.values.reshape(-1, 1)).ravel()
        df[col] = df.groupby("date")[col].transform(lam)



    return out


def scale_chars_cross_sectionally_by_month_bigdata_v2(
    df: pd.DataFrame,
    characteristic_cols: list[str],
    n_quantiles: int = 1000,
    random_state: int = 42,
    subsample: int = 10000,
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Scale specified columns cross-sectionally within each calendar month (year, month),
    using a separate QuantileTransformer per (month, column) to keep memory bounded.

    Assumes df has a datetime-like 'date' column by default; can be overridden with date_col.
    Returns a new DataFrame with transformed columns.
    """
    if date_col not in df.columns:
        raise KeyError(f"DataFrame must contain a '{date_col}' column.")

    out = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(out[date_col]):
        out[date_col] = pd.to_datetime(out[date_col])

    # Ensure we have month-key for grouping (YYYYMM)
    out["_ym"] = out[date_col].dt.to_period("M")

    # Cache transformers per (ym, col) if you want consistent scaling across months;
    # here we keep a per-group transformer to guarantee independence across months.
    transformers = {}

    for col in characteristic_cols:
        transformed = np.full(len(out), np.nan, dtype=float)

        for ym, grp in out.groupby("_ym", sort=False):
            idx = grp.index
            X = grp[[col]].values  # shape (n_in_group, 1)
            if X.size == 0 or np.all(pd.isna(X)):
                continue

            key = (str(ym), col)
            # per-month, per-column transformer
            if key not in transformers:
                n_q = max(2, min(n_quantiles, int(X.shape[0])))
                transformers[key] = QuantileTransformer(
                    n_quantiles=n_q,
                    output_distribution="uniform",
                    random_state=random_state,
                    subsample=subsample,
                )
            qt = transformers[key]

            X_tr = qt.fit_transform(X)
            transformed[idx] = X_tr[:, 0]

        # map to [-1, 1] if desired: uncomment below
        transformed = 2.0 * transformed - 1.0

        out[col] = transformed

    # drop helper column
    out.drop(columns=["_ym"], inplace=True)

    return out

In [7]:
# Increase this to make the benchmark heavier.
n_years = 60
n_firms_per_month = 5_000
seed = 42

rng = np.random.default_rng(seed)
months = pd.period_range("1960-01", periods=n_years * 12, freq="M")
date = np.repeat(months, n_firms_per_month)

df = pd.DataFrame({
    "date": pd.Series(date, dtype="period[M]"),
    "characteristic_groupby": rng.standard_normal(len(date)),
})
df["characteristic_loop"] = df["characteristic_groupby"].copy()

print(f"Rows: {len(df):,}")
print(f"Months: {df['date'].nunique():,}")
print(f"Firms per month: {n_firms_per_month:,}")
print(df.dtypes)
df.head()

Rows: 3,600,000
Months: 720
Firms per month: 5,000
date                      period[M]
characteristic_groupby      float64
characteristic_loop         float64
dtype: object


,date,characteristic_groupby,characteristic_loop
0,1960-01,0.304717,0.304717
1,1960-01,-1.039984,-1.039984
2,1960-01,0.750451,0.750451
3,1960-01,0.940565,0.940565
4,1960-01,-1.951035,-1.951035


In [8]:
def benchmark(func, column: str, repeats: int = 3) -> tuple[pd.DataFrame, list[float]]:
    times = []
    output = None

    for _ in range(repeats):
        start = time.perf_counter()
        output = func(
            df,
            characteristic_cols=[column],
            n_quantiles=1000,
            random_state=42,
        )
        times.append(time.perf_counter() - start)
        gc.collect()

    return output, times


In [9]:

result_groupby, times_groupby = benchmark(
    scale_chars_cross_sectionally_by_month,
    column="characteristic_groupby",
)


In [10]:

result_loop, times_loop = benchmark(
    scale_chars_cross_sectionally_by_month_bigdata_v2,
    column="characteristic_loop",
)


TypeError: Passing PeriodDtype data is invalid. Use `data.to_timestamp()` instead

In [ ]:

summary = pd.DataFrame({
    "function": [
        "scale_chars_cross_sectionally_by_month",
        "scale_chars_cross_sectionally_by_month_bigdata_v2",
    ],
    "median_seconds": [median(times_groupby), median(times_loop)],
    "min_seconds": [min(times_groupby), min(times_loop)],
    "max_seconds": [max(times_groupby), max(times_loop)],
    "all_runs_seconds": [times_groupby, times_loop],
})
summary["relative_to_fastest"] = summary["median_seconds"] / summary["median_seconds"].min()
summary

In [ ]:
# Correctness checks: the two identical input columns should yield identical scaled values.
scaled_groupby = result_groupby["characteristic_groupby"].to_numpy()
scaled_loop = result_loop["characteristic_loop"].to_numpy()

assert np.allclose(scaled_groupby, scaled_loop, equal_nan=True)
assert np.nanmin(scaled_groupby) >= -1.0
assert np.nanmax(scaled_groupby) <= 1.0

monthly_bounds = (
    result_groupby.groupby("date", observed=True)["characteristic_groupby"]
    .agg(["min", "max"])
)

print("Outputs match and lie in [-1, 1].")
monthly_bounds.head()